# Day 019 — Exercise 5: summarize_results and print_report

**Goal:** Implement `summarize_results(results)` and `print_report(results)`. These are pure functions — all checks use synthetic EvalResult objects, no Ollama calls needed.

In [ ]:
import re, ollama
from dataclasses import dataclass, field

def exact_match(response: str, expected: str) -> bool:
    return response.strip().lower() == expected.strip().lower()

def contains_any(response: str, keywords: list[str]) -> bool:
    resp_lower = response.lower()
    return any(kw.lower() in resp_lower for kw in keywords)

JUDGE_PROMPT = """\
You are an evaluation judge. Score the response below on a scale of 1 to 5.

Question: {question}
Expected answer: {expected}
Actual response: {response}

Rubric:
1 = Completely wrong or irrelevant
2 = Mostly wrong with minor correct elements
3 = Partially correct but with significant gaps
4 = Mostly correct with minor issues
5 = Fully correct and complete

Respond with ONLY this format:
Score: <1-5>
Rationale: <one sentence>
"""

def llm_judge(question, response, expected, model='llama3.2'):
    prompt = JUDGE_PROMPT.format(question=question, expected=expected, response=response)
    raw = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    text = raw['message']['content']
    m = re.search(r'Score:\s*([1-5])', text)
    score = int(m.group(1)) if m else 3
    r = re.search(r'Rationale:\s*(.+)', text)
    rationale = r.group(1).strip() if r else text.strip()[:200]
    return {'score': score, 'rationale': rationale}

@dataclass
class TestCase:
    question: str
    expected_keywords: list[str] = field(default_factory=list)
    expected_answer: str = ''

@dataclass
class EvalResult:
    test_case: TestCase
    response: str
    passed: bool
    matched_keywords: list[str] = field(default_factory=list)
    judge_score: int = 0
    judge_rationale: str = ''

def run_eval(test_cases, system_prompt='You are a helpful assistant.',
             model='llama3.2', use_judge=False):
    results = []
    for tc in test_cases:
        raw = ollama.chat(model=model, messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': tc.question},
        ])
        response = raw['message']['content']
        matched = [kw for kw in tc.expected_keywords
                   if kw.lower() in response.lower()]
        passed = bool(matched) if tc.expected_keywords else True
        result = EvalResult(tc, response, passed, matched)
        if use_judge and tc.expected_answer:
            j = llm_judge(tc.question, response, tc.expected_answer, model)
            result.judge_score = j['score']
            result.judge_rationale = j['rationale']
        results.append(result)
    return results


## Your Implementation

In [ ]:
def summarize_results(results: list[EvalResult]) -> dict:
    """
    Aggregate a list of EvalResults into a summary dict.

    Returns:
        {'total': int, 'passed': int, 'failed': int,
         'pass_rate': float, 'avg_judge_score': float}

    Returns all zeros for an empty list.
    avg_judge_score averages only results where judge_score > 0.
    """
    # TODO: handle empty list (return all zeros)
    # TODO: compute total, passed, failed
    # TODO: pass_rate = round(passed / total, 4)
    # TODO: avg_judge_score from results with judge_score > 0
    pass


def print_report(results: list[EvalResult]) -> None:
    """
    Print a formatted eval report with summary and per-case breakdown.
    """
    # TODO: call summarize_results, print header with total/pass_rate
    # TODO: for each result: print ✅/❌, question (truncated), and judge info if present
    pass


## Check Your Work

In [ ]:
import io, sys

def _make_result(passed, score=0):
    tc = TestCase('Q?', expected_keywords=['yes'])
    return EvalResult(tc, 'yes' if passed else 'no', passed,
                      judge_score=score)

def _run_checks():
    total = 5
    passed = 0

    # Check 1: both functions defined
    try:
        assert 'summarize_results' in globals() and 'print_report' in globals()
        passed += 1; print('✅ Check 1: summarize_results and print_report defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: empty list returns all zeros
    try:
        s = summarize_results([])
        assert s['total'] == 0 and s['passed'] == 0 and s['pass_rate'] == 0.0
        passed += 1; print('✅ Check 2: empty list returns all zeros')
    except Exception as e:
        print(f'❌ Check 2: empty list — {e}')

    # Check 3: all-passed results → pass_rate 1.0
    try:
        s = summarize_results([_make_result(True), _make_result(True)])
        assert s['total'] == 2 and s['passed'] == 2 and s['pass_rate'] == 1.0
        passed += 1; print('✅ Check 3: all-passed → pass_rate 1.0')
    except Exception as e:
        print(f'❌ Check 3: all passed — {e}')

    # Check 4: mixed results — correct totals and pass_rate
    try:
        s = summarize_results([_make_result(True), _make_result(False),
                               _make_result(True), _make_result(False)])
        assert s['total'] == 4 and s['passed'] == 2 and s['failed'] == 2
        assert abs(s['pass_rate'] - 0.5) < 1e-6
        passed += 1; print('✅ Check 4: mixed results — totals and pass_rate correct')
    except Exception as e:
        print(f'❌ Check 4: mixed results — {e}')

    # Check 5: avg_judge_score averages only non-zero scores
    try:
        s = summarize_results([
            _make_result(True, score=4),
            _make_result(True, score=0),   # judge did not run
            _make_result(False, score=2),
        ])
        assert abs(s['avg_judge_score'] - 3.0) < 0.01, \
            f'expected avg 3.0 from scores [4,2], got {s["avg_judge_score"]}'
        passed += 1; print('✅ Check 5: avg_judge_score averages only non-zero scores')
    except Exception as e:
        print(f'❌ Check 5: avg_judge_score — {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()


## Solution

<details>
<summary>Click to reveal</summary>

```python
def summarize_results(results):
    if not results:
        return {'total': 0, 'passed': 0, 'failed': 0,
                'pass_rate': 0.0, 'avg_judge_score': 0.0}
    total  = len(results)
    passed = sum(1 for r in results if r.passed)
    scores = [r.judge_score for r in results if r.judge_score > 0]
    return {
        'total': total,
        'passed': passed,
        'failed': total - passed,
        'pass_rate': round(passed / total, 4),
        'avg_judge_score': round(sum(scores) / len(scores), 2) if scores else 0.0,
    }

def print_report(results):
    s = summarize_results(results)
    print(f"Eval — {s['total']} cases | "
          f"Pass: {s['passed']}/{s['total']} ({s['pass_rate']*100:.1f}%)")
    if s['avg_judge_score'] > 0:
        print(f"  Avg judge score: {s['avg_judge_score']:.1f}/5")
    for i, r in enumerate(results, 1):
        icon = '✅' if r.passed else '❌'
        print(f"  {icon} {i}. {r.test_case.question[:55]}")
        if r.matched_keywords:
            print(f"     Keywords: {r.matched_keywords}")
        if r.judge_score:
            print(f"     Score {r.judge_score}/5 — {r.judge_rationale[:60]}")
```

</details>